# Đề Tài: Ứng Dụng BiLSTM Kết Hợp Cơ Chế Attention Trong Định Tuyến Hỗ Trợ Khách Hàng Đa Kênh

**Bài toán:** Phân loại theo nội dung độc lập với thương hiệu và kênh (Email dài, Live Chat, Ticket).
- Dự đoán **7 Loại vấn đề phổ quát** & **4 Mức ưu tiên**.
- Tích hợp **Cổng kiểm soát rủi ro (Human Triage Gate)**.
- Giải thích trực quan bằng **Bản đồ nhiệt Attention (Explainability Heatmap)**.

In [ ]:
import os
import sys
import torch
import matplotlib.pyplot as plt
from PIL import Image

sys.path.append('src')
from predict_universal import UniversalRouter

print('PyTorch Version:', torch.__version__)
print('Device:', 'cuda' if torch.cuda.is_available() else 'cpu')

## 1. Khởi tạo Universal Router & Nạp Mô Hình BiLSTM + Attention

In [ ]:
router = UniversalRouter(
    model_path='models/best_universal_bilstm_attention.pt',
    data_dir='data'
)
print('✅ Nạp Universal Router thành công!')

## 2. Kiểm thử Định tuyến & Trực quan hóa Trọng số Attention

In [ ]:
def inspect_ticket(text, industry='ecommerce', threshold=0.60):
    res = router.route_ticket(text, confidence_threshold=threshold, selected_industry=industry)
    print('Văn bản yêu cầu   :', res['raw_text'])
    print('Loại vấn đề       :', f"{res['category']['icon']} {res['category']['name_vi']} ({res['category']['confidence']*100:.2f}%)")
    print('Mức ưu tiên       :', f"{res['priority']['badge']} {res['priority']['name_vi']}")
    print('Trạng thái xử lý  :', f"[{res['triage']['status']}]")
    if res['triage']['human_review_required']:
        for reason in res['triage']['reasons']:
            print('  ⚠️', reason)
    print('Hàng đợi tiếp nhận:', f"{res['routing']['target_queue']} ({res['routing']['industry']})")
    
    tokens = [t['token'] for t in res['token_attentions']]
    weights = [t['weight'] for t in res['token_attentions']]
    
    plt.figure(figsize=(max(8, len(tokens)*0.55), 3.5))
    plt.bar(range(len(tokens)), weights, color='darkorange', edgecolor='brown', alpha=0.85)
    plt.xticks(range(len(tokens)), tokens, rotation=45, ha='right', fontsize=11)
    plt.ylabel('Trọng số Attention', fontsize=11)
    plt.title(f"{res['category']['name_vi']} ({res['category']['confidence']*100:.1f}%) | {res['priority']['name_vi']}", fontsize=12, fontweight='bold')
    plt.grid(axis='y', linestyle=':', alpha=0.6)
    plt.tight_layout()
    plt.show()

# Ca 1: Email thanh toán
inspect_ticket('Dear support team, my credit card was charged twice for the renewal fee, please issue a full refund immediately', industry='ecommerce')
# Ca 2: Khiếu nại gay gắt rủi ro cao
inspect_ticket('Your staff refused to assist and hung up. I will call the police and hire a lawyer for lawsuit!', industry='fintech')

## 3. Xem Ma Trận Nhầm Lẫn & Đường Cong Tỷ Lệ Tự Động Hóa (Risk-Coverage)

In [ ]:
img = Image.open('reports/universal_evaluation_curves.png')
plt.figure(figsize=(15, 6))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()